# Train and evaluate classifier

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from pathlib import Path
import evaluate
from datasets import Dataset, DatasetDict
from transformers import AdamW
from tqdm.auto import tqdm
from transformers import get_scheduler
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
clause_type = 'anti-scraping' # 'modification' |'opt-out'  |'arbitration' | 'class waiver' | 'anti-scraping'


In [ ]:

# processed_data_dir = Path('processed_data/tous')
# metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')

In [ ]:
df_annotations = pd.read_csv(f'annotations/ollama_annotations/{clause_type}_labels_judged.csv')


In [ ]:

df_annotations.rename({'label_gpt-4o':'labels','sentence':'text'}, axis=1, inplace=True)
df_annotations.head(3)


In [ ]:
df_annotations.value_counts('labels')

## Train the model

In [ ]:

from sklearn.model_selection import train_test_split

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

train,test = train_test_split(df_annotations, test_size=0.1, random_state=42)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train':  Dataset.from_pandas(train[['text','labels']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test[['text','labels']].reset_index(drop=True)),
})

dataset


In [ ]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_datasets

In [ ]:

tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

In [ ]:
tokenized_datasets

In [ ]:
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=2, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=4, collate_fn=data_collator
)

In [ ]:
import torch
device = (
        "cuda"
        if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available() else "cpu"
        )
#device = 'cpu'
print(device)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.to(device)
best_f1_score = .0


optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-6, weight_decay=0.2)



num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)



progress_bar = tqdm(range(num_training_steps))
metric = evaluate.load("glue", "mrpc")

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)




    model.eval()
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        metric.add_batch(predictions=predictions, references=batch["labels"])

    scores = metric.compute()
    print(f"Epoch {epoch}:", scores)
    print(f"Epoch {epoch}:", scores['f1'])

    if scores['f1'] > best_f1_score:
        print('Saving best model')
        model.save_pretrained(f"./models/{clause_type}_best_model_ft_ds")
        tokenizer.save_pretrained(f"./models/{clause_type}_best_model_ft_ds")
        best_f1_score = scores['f1']

In [ ]:
# model.save_pretrained(f"./models/{clause_type}_model_ft_ds")
# tokenizer.save_pretrained(f"./models/{clause_type}_model_ft_ds")

# Apply Classifier

In [ ]:
from torch.nn.functional import softmax
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
processed_data_dir = Path('processed_data/tous')
file_name = processed_data_dir / 'metadata_annotated_distilbert.tsv'
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

In [ ]:
metadata.columns

In [ ]:
clause_type = 'opt-out' #'arbitration' | 'opt-out' | 'class waiver'
model = AutoModelForSequenceClassification.from_pretrained(f'models/{clause_type}_model')
tokenizer = AutoTokenizer.from_pretrained(f'models/{clause_type}_model')

In [ ]:
#model.to('mps')

In [ ]:
metadata.head()

In [ ]:
metadata['sentence_processed_2'] = metadata.sentence.str.lower().str.strip()
mapping = metadata.groupby("sentence_processed_2").apply(lambda x: x.index.tolist()).to_dict()
len(mapping)

In [ ]:
sent2class = {s: softmax(model(**tokenizer(s, return_tensors='pt', truncation=True)).logits.detach(), dim=1)   
                         for s in tqdm(mapping.keys())}

In [ ]:
tqdm.pandas()
metadata[f'logits_{clause_type}']  = metadata.sentence_processed_2.progress_apply(lambda x: sent2class[x])
# metadata[f'logits_{clause_type}'] = metadata.progress_apply(lambda x: 
#                     softmax(model(**tokenizer(x.sentence, return_tensors='pt', truncation=True).to('mps')).logits.detach(), dim=1), 
#                           axis=1)



In [ ]:
metadata[f'prob_1_{clause_type}'] = metadata[f'logits_{clause_type}'].apply(lambda x: x[0][1].item())
metadata[clause_type] = .0
metadata.loc[metadata[f'prob_1_{clause_type}'] > .5, clause_type] = 1

In [ ]:
file_name

In [ ]:
metadata.to_csv(processed_data_dir /f'metadata_annotated_distilbert.tsv', sep='\t', index=False)

In [ ]:
metadata[clause_type].value_counts()

In [ ]:
metadata.columns

In [ ]:
metadata['year_int'] = metadata.year.apply(lambda x: int(str(x)[:4]))

In [ ]:
import seaborn as sns

data = metadata.groupby(['platform','year_int'])[clause_type].sum().astype(bool).astype(int).unstack().fillna(-1)



#.loc['bumble'].plot(kind='bar')

In [ ]:
import pandas as _pd

def replace_minus_ones_with_prev(X, axis=1, inplace=False):
    """
    Replace -1 entries in a matrix/array with the nearest preceding 0 or 1 along the given axis.
    If there is no preceding non -1 value, the -1 is left unchanged.

    Parameters:
    - X: array-like (numpy array, list of lists, or pandas DataFrame)
    - axis: 1 to replace along rows (left-to-right), 0 to replace along columns (top-to-bottom)
    - inplace: if True and X is a numpy array or DataFrame, modify it in place; otherwise return a new array

    Returns:
    - numpy.ndarray or pandas.DataFrame with replacements applied (unless inplace=True modifies input)
    """


    is_df = _pd is not None and isinstance(X, _pd.DataFrame)
    if is_df:
        arr = X.values
    else:
        arr = X if isinstance(X, (np.ndarray,)) else np.array(X)

    if not inplace:
        arr = arr.copy()

    if axis not in (0, 1):
        raise ValueError("axis must be 0 or 1")

    # iterate over the chosen axis and carry forward the last seen non -1 value
    if axis == 1:
        # rows
        for r in range(arr.shape[0]):
            last = None
            for c in range(arr.shape[1]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last
    else:
        # columns
        for c in range(arr.shape[1]):
            last = None
            for r in range(arr.shape[0]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last

    if is_df:
        if inplace:
            X.iloc[:, :] = arr
            return X
        else:
            return _pd.DataFrame(arr, index=X.index, columns=X.columns)
    else:
        return arr

sns.set(rc={'figure.figsize':(6.7,10.27)})
sns.heatmap(replace_minus_ones_with_prev(data),cbar=False)

In [ ]:
metadata.columns

In [ ]:
#metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.columns

In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence'])
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.text)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:


    df_deduplicated.loc[df_deduplicated.prob_1.between(*interval),'category']  = label



In [ ]:
pd.concat([df_deduplicated[df_deduplicated.category == label].sample(10)
    for _ , label in int_labels], axis=0)[['sentence','category']].to_csv(f'annotations/inference/{clause_type}_automatic_annotations_by_category.csv')


In [ ]:
metadata[metadata.prob_1 > .5].to_csv(f'annotations/inference/{clause_type}_inference.csv')

## Fin